### 1. Carga y Auditoría Inicial de los Datos

Se están importando los cinco ficheros CSV previamente limpiados (`Meteorología`, `Distritos`, `Socioeconomía`, `Calendario` y `Activaciones`), se almacenan en un diccionario centralizado en memoria y hacemos diagnóstico estructural de cada uno (dimensiones, nombres de columnas y primeras 3 filas).

Antes de realizar agregaciones o cruces, es fundamental verificar que todos los archivos cargan sin errores de codificación o ruta, identificar la granularidad temporal y espacial de cada tabla, y detectar qué columnas servirán como claves de unión (*keys*) o requerirán limpieza adicional.

In [2]:
import pandas as pd

# Rutas de los archivos originales
ruta_base = r"C:\Users\bario\OneDrive\Documentos\UC3M\PROYECTO EN CIENCIA DE DATOS\proyecto_emergencias-20260915T155207Z-1-001\proyecto_emergencias\datasets limpios"
archivos = {
    "Meteorología": "meteo_limpio.csv",
    "Distritos": "distritos_limpio.csv",
    "Socioeconomía": "socioeconomico_limpio.csv",
    "Calendario": "calendario_limpio.csv",
    "Activaciones": "activaciones_limpio (1).csv"
}

datasets = {}

for nombre, archivo in archivos.items():

    df = pd.read_csv(f"{ruta_base}\\{archivo}")
    datasets[nombre] = df

    print("\n" + "=" * 70)
    print(f"{nombre.upper()}")
    print("=" * 70)

    print("Dimensiones:", df.shape)
    print("\nColumnas:")
    print(df.columns.tolist())

    print("\nPrimeras 3 filas:")
    print(df.head(3).to_string(index=False))


METEOROLOGÍA
Dimensiones: (366, 10)

Columnas:
['fecha', 'temperatura_max', 'temperatura_min', 'temperatura_media', 'precipitacion_mm', 'humedad_relativa', 'viento_max_kmh', 'presion_hpa', 'indice_uv', 'condicion']

Primeras 3 filas:
     fecha  temperatura_max  temperatura_min  temperatura_media  precipitacion_mm  humedad_relativa  viento_max_kmh  presion_hpa  indice_uv condicion
01-01-2024             11.6              0.8                6.4               0.0              40.0            19.0       1010.2          1 DESPEJADO
02-01-2024              9.7              0.1               -0.0               0.1              32.0             8.5       1016.5          1 DESPEJADO
03-01-2024             12.0              0.4                8.2               0.0              38.0             9.7       1031.1          2   NUBLADO

DISTRITOS
Dimensiones: (21, 7)

Columnas:
['distrito_id', 'nombre', 'codigo_postal_principal', 'poblacion', 'superficie_km2', 'num_hospitales', 'base_samur']

Prime

**Análisis**
* **Meteorología `(366, 10)`:** Granularidad diaria completa para el año bisiesto 2024. Contiene tanto métricas continuas (temperaturas, precipitación, viento, presión) como una variable categórica (`condicion`: *DESPEJADO*, *NUBLADO*, etc.).
* **Distritos `(21, 7)`:** Granularidad espacial estática con los 21 distritos de Madrid. Incluye identificadores (`distrito_id`, `nombre`) e infraestructura sanitaria (`num_hospitales`, `base_samur`).
* **Socioeconomía `(21, 15)`:** Misma granularidad espacial (21 distritos). Aporta indicadores demográficos y sociales clave (`renta_media_hogar`, `pct_mayores_65`, `indice_vulnerabilidad`), pero repite columnas ya presentes en `Distritos` (`distrito_id`, `poblacion_2024`, `superficie_km2`) que habrá que filtrar al cruzarlas para evitar redundancias.
* **Calendario `(366, 14)`:** Granularidad diaria (366 días de 2024). Detalla el contexto temporal y social de Madrid (festivos, vísperas, vacaciones escolares, eventos y estaciones).
* **Activaciones `(146.567, 13)`:** Registro transaccional con 146.567 intervenciones de emergencia en 2024. Carece de fecha exacta (`YYYY-MM-DD`), pero dispone de `año`, `mes` (en texto mayúsculas) y `hora_solicitud` (siempre informada, a diferencia de `hora_intervencion` o `tiempo_de_respuesta` que presentan `NaN` cuando no se llega a intervenir). Esto confirma que la construcción del dataset final a nivel **mes × distrito × franja horaria** es totalmente viable.

### 2. Construcción de la Matriz Mensual de Características ($X$)

Creamos la función `construir_matriz_x_mensual(datasets)` para consolidar todas las variables explicativas (meteorología, calendario, geografía y socioeconomía) en una única tabla con granularidad **mes × distrito** (12 meses × 21 distritos = 252 filas):
1. **Agregación meteorológica:** Se agrupan los registros diarios por mes calculando estadísticos descriptivos (medias, máximos, mínimos y acumulados) y se contabilizan los días según la condición atmosférica (`DESPEJADO`, `LLUVIA`, `NUBLADO`, `TORMENTA`) mediante codificación *one-hot* (`pd.get_dummies`).
2. **Agregación de calendario:** Se transforman en indicadores binarios los fines de semana, festivos, eventos y vacaciones escolares, sumándolos por mes junto con el desglose de tipos de día, estaciones y tipos de evento.
3. **Consolidación territorial (Distritos + Socioeconomía):** Se estandarizan los nombres de los 21 distritos, se verifica su coincidencia exacta y se cruzan ambas tablas eliminando columnas redundantes (`distrito_id`, `poblacion_2024`, `superficie_km2`).
4. **Producto cartesiano y control de calidad:** Se genera la rejilla completa de 252 combinaciones mes-distrito mediante `pd.MultiIndex.from_product()`, se incorporan las variables temporales y territoriales, y se aplican validaciones estrictas (`assert`) de dimensionalidad, unicidad y ausencia de nulos.

**¿Por qué se hace?**
Como el registro de `Activaciones` solo dispone del mes y no del día exacto, trabajar a nivel diario generaría una matriz artificial de 7.686 filas (366 días × 21 distritos) sin correspondencia directa con la variable objetivo ($y$). Agregar directamente a nivel mensual evita crear registros innecesarios, captura la estacionalidad climática y social de cada mes y garantiza una base consolidada sin duplicados para el posterior cruce con las franjas horarias.


In [3]:

import pandas as pd
import numpy as np

# ==============================================================================
# PASO 2: CONSTRUCCIÓN DIRECTA DE LA MATRIZ MENSUAL DE CARACTERÍSTICAS (X)
# ==============================================================================

def construir_matriz_x_mensual(datasets):

    # --------------------------------------------------------------------------
    # 1. CARGAR LAS FUENTES
    # --------------------------------------------------------------------------

    df_meteo = datasets['Meteorología'].copy()
    df_distritos = datasets['Distritos'].copy()
    df_socio = datasets['Socioeconomía'].copy()
    df_cal = datasets['Calendario'].copy()

    meses = {
        1: 'ENERO', 2: 'FEBRERO', 3: 'MARZO',
        4: 'ABRIL', 5: 'MAYO', 6: 'JUNIO',
        7: 'JULIO', 8: 'AGOSTO', 9: 'SEPTIEMBRE',
        10: 'OCTUBRE', 11: 'NOVIEMBRE', 12: 'DICIEMBRE'
    }

    # Convertir fechas y extraer el mes
    for df in [df_meteo, df_cal]:
        df['fecha'] = pd.to_datetime(
            df['fecha'], format='%d-%m-%Y', errors='raise'
        )
        df['mes_num'] = df['fecha'].dt.month

    # --------------------------------------------------------------------------
    # 2. AGREGACIÓN MENSUAL DE LAS VARIABLES METEOROLÓGICAS
    # --------------------------------------------------------------------------

    meteo_mensual = df_meteo.groupby('mes_num').agg(
        temperatura_max_media=('temperatura_max', 'mean'),
        temperatura_max_absoluta=('temperatura_max', 'max'),
        temperatura_min_media=('temperatura_min', 'mean'),
        temperatura_min_absoluta=('temperatura_min', 'min'),
        temperatura_media=('temperatura_media', 'mean'),
        precipitacion_total=('precipitacion_mm', 'sum'),
        precipitacion_maxima=('precipitacion_mm', 'max'),
        humedad_media=('humedad_relativa', 'mean'),
        viento_medio=('viento_max_kmh', 'mean'),
        viento_maximo=('viento_max_kmh', 'max'),
        presion_media=('presion_hpa', 'mean'),
        indice_uv_medio=('indice_uv', 'mean'),
        indice_uv_maximo=('indice_uv', 'max'),
        dias_registrados=('fecha', 'count')
    ).reset_index()

    # Contar los días de cada condición meteorológica
    condiciones = pd.get_dummies(
        df_meteo['condicion'], prefix='dias_condicion', dtype=int
    )

    condiciones['mes_num'] = df_meteo['mes_num'].values

    condiciones_mensuales = (
        condiciones.groupby('mes_num').sum().reset_index()
    )

    meteo_mensual = meteo_mensual.merge(
        condiciones_mensuales, on='mes_num', validate='one_to_one'
    )

    # --------------------------------------------------------------------------
    # 3. AGREGACIÓN MENSUAL DE LAS VARIABLES DEL CALENDARIO
    # --------------------------------------------------------------------------

    df_cal['es_fin_semana'] = (
        df_cal['dia_semana_num'].isin([5, 6]).astype(int)
    )

    df_cal['es_festivo'] = (
        df_cal['festivo'].ne('NO_FESTIVO').astype(int)
    )

    df_cal['es_dia_con_evento'] = (
        df_cal['evento'].ne('SIN_EVENTO').astype(int)
    )

    df_cal['es_vacaciones'] = (
        df_cal['vacaciones_escolares'].astype(int)
    )

    calendario_mensual = df_cal.groupby('mes_num').agg(
        dias_mes=('fecha', 'count'),
        dias_fin_semana=('es_fin_semana', 'sum'),
        dias_festivos=('es_festivo', 'sum'),
        dias_con_evento=('es_dia_con_evento', 'sum'),
        dias_vacaciones=('es_vacaciones', 'sum'),
        dias_vispera_festivo=('es_vispera_festivo', 'sum')
    ).reset_index()

    # Contar los días según tipo de día, estación y tipo de evento
    for columna in ['tipo_dia', 'estacion', 'tipo_evento']:

        indicadores = pd.get_dummies(
            df_cal[columna],
            prefix=f'dias_{columna}',
            dtype=int
        )

        indicadores['mes_num'] = df_cal['mes_num'].values

        indicadores_mensuales = (
            indicadores.groupby('mes_num').sum().reset_index()
        )

        calendario_mensual = calendario_mensual.merge(
            indicadores_mensuales,
            on='mes_num',
            validate='one_to_one'
        )

    # --------------------------------------------------------------------------
    # 4. UNIR LAS VARIABLES MENSUALES DE METEOROLOGÍA Y CALENDARIO
    # --------------------------------------------------------------------------

    caracteristicas_mensuales = pd.merge(
        meteo_mensual,
        calendario_mensual,
        on='mes_num',
        how='inner',
        validate='one_to_one'
    )

    # --------------------------------------------------------------------------
    # 5. INTEGRAR LAS VARIABLES ESTÁTICAS DE LOS DISTRITOS
    # --------------------------------------------------------------------------

    # Normalizar los nombres para facilitar el cruce
    df_distritos['nombre'] = (
        df_distritos['nombre'].astype(str).str.strip().str.upper()
    )

    df_socio['distrito'] = (
        df_socio['distrito'].astype(str).str.strip().str.upper()
    )

    # Comprobar que los 21 distritos coinciden
    assert set(df_distritos['nombre']) == set(df_socio['distrito']), (
        "Error: Los distritos no coinciden entre ambas fuentes."
    )

    # Evitar duplicar las variables que ya existen en distritos
    columnas_socio = [
        c for c in df_socio.columns
        if c not in [
            'distrito', 'distrito_id',
            'poblacion_2024', 'superficie_km2'
        ]
    ]

    distrito_info = pd.merge(
        df_distritos,
        df_socio[columnas_socio + ['distrito']],
        left_on='nombre',
        right_on='distrito',
        how='inner',
        validate='one_to_one'
    )

    distrito_info = distrito_info.drop(columns=['distrito'])

    distrito_info = distrito_info.rename(
        columns={'nombre': 'distrito'}
    )

    # --------------------------------------------------------------------------
    # 6. CREAR DIRECTAMENTE LAS 252 COMBINACIONES MES-DISTRITO
    # --------------------------------------------------------------------------

    meses_disponibles = sorted(caracteristicas_mensuales['mes_num'].unique())

    distritos_disponibles = distrito_info['distrito'].unique()

    matriz_x = pd.MultiIndex.from_product(
        [meses_disponibles, distritos_disponibles],
        names=['mes_num', 'distrito']
    ).to_frame(index=False)

    # Incorporar características mensuales
    matriz_x = matriz_x.merge(
        caracteristicas_mensuales,
        on='mes_num',
        how='left',
        validate='many_to_one'
    )

    # Incorporar características de cada distrito
    matriz_x = matriz_x.merge(
        distrito_info,
        on='distrito',
        how='left',
        validate='many_to_one'
    )

    # Crear el nombre del mes
    matriz_x['mes'] = matriz_x['mes_num'].map(meses)

    # Ordenar las columnas principales
    primeras_columnas = ['mes', 'mes_num', 'distrito', 'distrito_id']

    otras_columnas = [
        c for c in matriz_x.columns if c not in primeras_columnas
    ]

    matriz_x = matriz_x[primeras_columnas + otras_columnas]

    # --------------------------------------------------------------------------
    # 7. VALIDACIONES
    # --------------------------------------------------------------------------

    assert len(matriz_x) == 252, (
        f"Dimensión incorrecta: {len(matriz_x)} filas en lugar de 252."
    )

    assert matriz_x.duplicated(
        subset=['mes', 'distrito']
    ).sum() == 0, "Se han encontrado combinaciones duplicadas."

    assert matriz_x.isna().sum().sum() == 0, (
        "Se han encontrado valores nulos en la matriz X."
    )

    assert matriz_x['mes'].nunique() == 12
    assert matriz_x['distrito'].nunique() == 21

    print("=" * 65)
    print("MATRIZ MENSUAL X CONSTRUIDA CORRECTAMENTE")
    print("=" * 65)
    print("Filas:", len(matriz_x))
    print("Columnas:", matriz_x.shape[1])
    print("Meses:", matriz_x['mes'].nunique())
    print("Distritos:", matriz_x['distrito'].nunique())
    print("Nulos:", matriz_x.isna().sum().sum())

    print("\nPrimeras filas:")
    print(matriz_x.head().to_string(index=False))

    print("\nVariables creadas:")
    print(matriz_x.columns.tolist())

    return matriz_x


# EJECUCIÓN
matriz_x_mensual = construir_matriz_x_mensual(datasets)


MATRIZ MENSUAL X CONSTRUIDA CORRECTAMENTE
Filas: 252
Columnas: 57
Meses: 12
Distritos: 21
Nulos: 0

Primeras filas:
  mes  mes_num   distrito  distrito_id  temperatura_max_media  temperatura_max_absoluta  temperatura_min_media  temperatura_min_absoluta  temperatura_media  precipitacion_total  precipitacion_maxima  humedad_media  viento_medio  viento_maximo  presion_media  indice_uv_medio  indice_uv_maximo  dias_registrados  dias_condicion_DESPEJADO  dias_condicion_LLUVIA  dias_condicion_NUBLADO  dias_condicion_TORMENTA  dias_mes  dias_fin_semana  dias_festivos  dias_con_evento  dias_vacaciones  dias_vispera_festivo  dias_tipo_dia_FESTIVO  dias_tipo_dia_FIN_DE_SEMANA  dias_tipo_dia_LABORABLE  dias_tipo_dia_PUENTE  dias_estacion_INVIERNO  dias_estacion_OTONO  dias_estacion_PRIMAVERA  dias_estacion_VERANO  dias_tipo_evento_CELEBRACION  dias_tipo_evento_CONCIERTO  dias_tipo_evento_DEPORTIVO  dias_tipo_evento_FIESTAS  dias_tipo_evento_SIN_EVENTO  codigo_postal_principal  poblacion  superfic

**Análisis**
* **Dimensionalidad:** La matriz resultante cuenta con **252 filas** (12 meses × 21 distritos) y **57 columnas**, con **0 valores nulos**.
* **Coherencia meteorológica y de calendario:** En las primeras filas correspondientes a `ENERO` se observa consistencia total entre filas del mismo mes: 31 días registrados (`dias_registrados` = `dias_mes` = 31), temperaturas máximas medias de `9,67 °C` (pico de `14,7 °C`), mínimas de hasta `-6,2 °C`, `59,5 mm` de precipitación acumulada, `16` días despejados, `8` días de fin de semana, `2` festivos y `31` días de invierno.
* **Diferenciación territorial:** Mientras las variables mensuales se mantienen constantes a lo largo de los distritos en enero, las métricas estáticas reflejan correctamente las particularidades de cada zona (por ejemplo, `SALAMANCA` presenta la mayor renta media del hogar con `71.432 €` y menor índice de vulnerabilidad `0,15`, frente a `ARGANZUELA` con `39.876 €` y vulnerabilidad `0,42`).

**Nota: ¿Por qué todos los distritos tienen la misma temperatura en un mismo mes?**

* **Por la estructura de los datos de origen:** El archivo `meteo_limpio.csv` cuenta con 366 filas (una por cada día de 2024) y carece de columna de `distrito`, al tratarse de una serie meteorológica única a nivel de ciudad (correspondiente a una estación de referencia de Madrid, como Retiro o Barajas). Al no disponer de registros separados por estación local, todos los distritos comparten las mismas condiciones climáticas dentro de cada mes.
* **Por el funcionamiento de los modelos predictivos :** Al entrenar un modelo para predecir el número de emergencias ($y$) en cada fila `(mes, distrito, franja)`:
  * Las **variables meteorológicas y de calendario** explican la **variación temporal** (por qué en enero o en julio hay más o menos emergencias en el conjunto de la ciudad, o cómo influyen la lluvia y las temperaturas extremas).
  * Las **variables socioeconómicas y de distrito** explican la **variación espacial** (por qué en un mismo mes de enero se registran más emergencias en Centro que en Retiro debido a diferencias en densidad de población, renta, envejecimiento o dotación hospitalaria).

### 3. Construcción de la Matriz Objetivo ($y$) por Franjas Horarias

Creamos la función `construir_matriz_y_franjas(datasets, matriz_x)` para transformar el registro individual de emergencias (`Activaciones`) en conteos agregados a nivel **mes × distrito × franja horaria** (ya que como hemos visto no podemos predecir por día a falta de la columna fecha en Activaciones):
1. **Validación temporal y normalización:** Se verifica que las 146.567 activaciones pertenezcan íntegramente al año 2024, y se estandarizan los textos de `mes` y `distrito` (mayúsculas y sin espacios sobrantes) comprobando que todos los meses coincidan con los de la matriz $X$.
2. **Discretización horaria:** Se extrae la hora entera a partir de `hora_solicitud` (`%H:%M:%S`) y se clasifica cada intervención en 4 franjas de 6 horas (`Madrugada`, `Mañana`, `Tarde`, `Noche`), etiquetando como `Desconocido` los casos en los que la hora sea nula o no tenga el formato esperado.
3. **Agregación y auditoría de integridad:** Se agrupan los registros por `(mes, distrito, franja_horaria)` contabilizando el volumen total (`total_intervenciones`), se verifica mediante `assert` que la suma de conteos coincida con las 146.567 filas originales, y se diagnostican posibles distritos fuera de los 21 oficiales.

Para construir la variable objetivo ($y$) con la granularidad operativa deseada (demanda por tramo horario del día) y detectar antes del cruce final cualquier anomalía en los datos transaccionales (como horas ausentes o ubicaciones no asignables a distritos administrativos) sin perder trazabilidad sobre el total de intervenciones reales.


In [4]:

# ==============================================================================
# PASO 3: CONSTRUCCIÓN DIRECTA DE LA MATRIZ OBJETIVO (y)
# ==============================================================================

def construir_matriz_y_franjas(datasets, matriz_x):

    # --------------------------------------------------------------------------
    # 1. CARGAR LAS ACTIVACIONES
    # --------------------------------------------------------------------------

    df_act = datasets['Activaciones'].copy()

    print("Registros originales:", len(df_act))

    # Comprobar el año de los registros
    df_act['año'] = pd.to_numeric(df_act['año'], errors='coerce')

    print("\nAños disponibles:")
    print(df_act['año'].value_counts(dropna=False).sort_index())

    assert df_act['año'].notna().all(), (
        "Hay registros sin año válido."
    )

    assert set(df_act['año'].unique()) == {2024}, (
        "Las activaciones no corresponden exclusivamente a 2024."
    )

    # --------------------------------------------------------------------------
    # 2. NORMALIZAR MESES Y DISTRITOS
    # --------------------------------------------------------------------------

    df_act['mes'] = (
        df_act['mes'].astype(str).str.strip().str.upper()
    )

    df_act['distrito'] = (
        df_act['distrito'].astype(str).str.strip().str.upper()
    )

    # Comprobar que los meses son válidos
    meses_validos = set(matriz_x['mes'].unique())

    assert set(df_act['mes'].unique()).issubset(meses_validos), (
        "Hay meses en activaciones que no aparecen en la matriz X."
    )

    # --------------------------------------------------------------------------
    # 3. EXTRAER LA HORA DE SOLICITUD
    # --------------------------------------------------------------------------

    df_act['hora_num'] = pd.to_datetime(
        df_act['hora_solicitud'],
        format='%H:%M:%S',
        errors='coerce'
    ).dt.hour

    # --------------------------------------------------------------------------
    # 4. CLASIFICAR LAS FRANJAS HORARIAS
    # --------------------------------------------------------------------------

    condiciones = [
        df_act['hora_num'].between(0, 5),
        df_act['hora_num'].between(6, 11),
        df_act['hora_num'].between(12, 17),
        df_act['hora_num'].between(18, 23)
    ]

    franjas = [
        '1. Madrugada (00:00 - 05:59)',
        '2. Mañana (06:00 - 11:59)',
        '3. Tarde (12:00 - 17:59)',
        '4. Noche (18:00 - 23:59)'
    ]

    df_act['franja_horaria'] = np.select(
        condiciones,
        franjas,
        default='Desconocido'
    )

    # --------------------------------------------------------------------------
    # 5. AGREGAR LAS ACTIVACIONES POR MES, DISTRITO Y FRANJA
    # --------------------------------------------------------------------------

    matriz_y = (
        df_act.groupby(
            ['mes', 'distrito', 'franja_horaria']
        )
        .size()
        .reset_index(name='total_intervenciones')
    )

    # --------------------------------------------------------------------------
    # 6. VALIDAR QUE NO SE HAN PERDIDO REGISTROS
    # --------------------------------------------------------------------------

    total_original = len(df_act)
    total_agregado = matriz_y['total_intervenciones'].sum()

    assert total_original == total_agregado, (
        f"Error: {total_original} registros originales frente a "
        f"{total_agregado} registros agregados."
    )

    # --------------------------------------------------------------------------
    # 7. COMPROBAR LOS DISTRITOS
    # --------------------------------------------------------------------------

    distritos_oficiales = set(matriz_x['distrito'].unique())
    distritos_activaciones = set(matriz_y['distrito'].unique())

    distritos_extra = distritos_activaciones - distritos_oficiales

    print("\n" + "=" * 65)
    print("MATRIZ OBJETIVO y CONSTRUIDA")
    print("=" * 65)

    print("Registros originales:", total_original)
    print("Grupos mes-distrito-franja:", len(matriz_y))
    print("Meses:", matriz_y['mes'].nunique())
    print("Distritos encontrados:", matriz_y['distrito'].nunique())

    print("\nIntervenciones por franja:")
    print(
        matriz_y.groupby('franja_horaria')['total_intervenciones']
        .sum()
    )

    print("\nDistritos que no pertenecen a los 21 oficiales:")
    print(sorted(distritos_extra))

    if len(distritos_extra) > 0:
        print("\nIntervenciones en esos distritos:")
        print(
            matriz_y[matriz_y['distrito'].isin(distritos_extra)]
            .groupby('distrito')['total_intervenciones']
            .sum()
        )

    print("\nTotal de intervenciones agregadas:", total_agregado)

    print("\nPrimeras filas:")
    print(matriz_y.head(10).to_string(index=False))

    return matriz_y


# EJECUCIÓN
matriz_y_franjas = construir_matriz_y_franjas(
    datasets,
    matriz_x_mensual
)


Registros originales: 146567

Años disponibles:
año
2024    146567
Name: count, dtype: int64

MATRIZ OBJETIVO y CONSTRUIDA
Registros originales: 146567
Grupos mes-distrito-franja: 1317
Meses: 12
Distritos encontrados: 24

Intervenciones por franja:
franja_horaria
1. Madrugada (00:00 - 05:59)    20562
2. Mañana (06:00 - 11:59)       31007
3. Tarde (12:00 - 17:59)        50621
4. Noche (18:00 - 23:59)        42955
Desconocido                      1422
Name: total_intervenciones, dtype: int64

Distritos que no pertenecen a los 21 oficiales:
['CARRETERAS Y CIRCUNVALACIONES', 'FUERA DE TERMINO', 'NO CONSTA']

Intervenciones en esos distritos:
distrito
CARRETERAS Y CIRCUNVALACIONES      1
FUERA DE TERMINO                  33
NO CONSTA                        567
Name: total_intervenciones, dtype: int64

Total de intervenciones agregadas: 146567

Primeras filas:
  mes   distrito               franja_horaria  total_intervenciones
ABRIL ARGANZUELA 1. Madrugada (00:00 - 05:59)                    

**Análisis**
* **Integridad total del conteo:** Se conservan exactamente las **146.567 intervenciones** originales agrupadas en **1.317 combinaciones** `(mes, distrito, franja_horaria)`.
* **Distribución horaria de la demanda:** La actividad muestra un claro patrón diurno-vespertino: la **Tarde (12:00–17:59)** concentra el mayor volumen con **50.621 intervenciones** (34,5 %), seguida de la **Noche (18:00–23:59)** con **42.955** (29,3 %), la **Mañana (06:00–11:59)** con **31.007** (21,2 %) y la **Madrugada (00:00–05:59)** con **20.562** (14,0 %).
* **Detección de dos anomalías clave para el Paso 4:**
  1. **Horas desconocidas (`Desconocido`):** Existen **1.422 activaciones** (0,97 % del total) sin hora de solicitud válida.
  2. **Ubicaciones fuera de los 21 distritos oficiales:** Aparecen **24 distritos** en lugar de 21. Los 3 extra corresponden a `NO CONSTA` (**567** casos), `FUERA DE TERMINO` (**33** casos) y `CARRETERAS Y CIRCUNVALACIONES` (**1** caso), sumando **601 intervenciones** (0,41 % del total) que no tienen correspondencia socioeconómica ni territorial en la matriz $X$.

#### 3.1. Agrupación de Distritos No Oficiales en la Matriz Objetivo ($y$)

Se crea una copia de `matriz_y_franjas` y se unifican las tres categorías espaciales ajenas a la división administrativa oficial (`CARRETERAS Y CIRCUNVALACIONES`, `FUERA DE TERMINO` y `NO CONSTA`) bajo una única etiqueta común: `'OTROS / DISTRITO NO IDENTIFICADO'`. Posteriormente, se re-agrupan los registros por `(mes, distrito, franja_horaria)` sumando el total de intervenciones y validando que se conserva el 100 % del volumen original.

Las tres categorías extra tienen un volumen residual (apenas 601 intervenciones entre las tres, destacando `CARRETERAS Y CIRCUNVALACIONES` con un único caso en todo el año) y carecen de variables socioeconómicas o de infraestructura asociadas en la matriz $X$. Agruparlas en una única categoría residual evita la fragmentación de datos innecesaria, mantiene la trazabilidad completa del total de emergencias y facilita su gestión en el paso de filtrado previo al modelado.


In [5]:
# Copiamos la matriz original
matriz_y_franjas_agrupada = matriz_y_franjas.copy()

# Agrupamos las tres categorías no oficiales
distritos_agrupados = [
    'CARRETERAS Y CIRCUNVALACIONES',
    'FUERA DE TERMINO',
    'NO CONSTA'
]

matriz_y_franjas_agrupada['distrito'] = (
    matriz_y_franjas_agrupada['distrito'].replace(
        distritos_agrupados,
        'OTROS / DISTRITO NO IDENTIFICADO'
    )
)

# Volvemos a agrupar por si varias categorías pasan a coincidir
matriz_y_franjas_agrupada = (
    matriz_y_franjas_agrupada
    .groupby(
        ['mes', 'distrito', 'franja_horaria'],
        as_index=False
    )['total_intervenciones']
    .sum()
)

# Comprobaciones
print("MATRIZ Y AGRUPADA")
print("=" * 50)

print("Grupos:", len(matriz_y_franjas_agrupada))
print("Distritos:", matriz_y_franjas_agrupada['distrito'].nunique())

print("\nIntervenciones totales:")
print(matriz_y_franjas_agrupada['total_intervenciones'].sum())

print("\nIntervenciones por distrito:")
print(
    matriz_y_franjas_agrupada.groupby('distrito')['total_intervenciones']
    .sum()
    .tail()
)

# Verificamos que no se ha perdido ninguna intervención
assert (
    matriz_y_franjas_agrupada['total_intervenciones'].sum()
    == 146567
)

MATRIZ Y AGRUPADA
Grupos: 1303
Distritos: 22

Intervenciones totales:
146567

Intervenciones por distrito:
distrito
TETUAN               7957
USERA                6149
VICALVARO            2481
VILLA DE VALLECAS    3805
VILLAVERDE           4943
Name: total_intervenciones, dtype: int64



**Análisis**
* **Reducción de categorías espaciales:** El número de distritos únicos pasa de **24 a 22** (los 21 distritos oficiales de Madrid más la nueva categoría agrupada `'OTROS / DISTRITO NO IDENTIFICADO'`).
* **Consolidación de filas:** El total de combinaciones `(mes, distrito, franja_horaria)` disminuye ligeramente de **1.317 a 1.303 grupos**, ya que las intervenciones de las tres categorías residuales que coincidían en el mismo mes y franja horaria se han sumado en una sola fila.
* **Conservación total de datos:** La aserción (`assert`) confirma que se mantienen intactas las **146.567 intervenciones** totales del año 2024 sin ninguna pérdida de información.

#### 3.2. Ampliación de la Matriz $X$ para Incorporar la Categoría Residual de Distrito

Se amplía la matriz mensual de características (`matriz_x_mensual`) creando 12 nuevas filas —una por cada mes del año— asignadas a la categoría `'OTROS / DISTRITO NO IDENTIFICADO'`, dando lugar a `matriz_x_ampliada`:
1. **Identificación de atributos territoriales:** Se listan las 17 variables específicas de distrito (demografía, socioeconomía, dotación sanitaria y escolar, superficie, etc.).
2. **Generación de filas mensuales para `OTROS`:** Para cada mes, se toma como plantilla una fila existente para conservar intactas las variables meteorológicas y de calendario (comunes a toda la ciudad), se asigna `'OTROS / DISTRITO NO IDENTIFICADO'` como distrito y se rellenan con `NaN` los 17 atributos territoriales.
3. **Concatenación y validación:** Se unen las 12 filas nuevas a las 252 originales y se verifica mediante aserciones (`assert`) que la dimensión final sea exactamente $12 \times 22 = 264$ filas sin combinaciones `(mes, distrito)` duplicadas.

¿Por qué?
En el paso anterior se agruparon 601 intervenciones sin distrito oficial bajo `'OTROS / DISTRITO NO IDENTIFICADO'`. Si no existiera esta categoría en la matriz $X$, esas 601 emergencias se perderían al realizar el cruce relacional (*merge*) entre $X$ e $y$. Al incorporar estas 12 filas con `NaN` en las variables espaciales, se preserva el 100 % de las intervenciones históricas sin inventar ni imputar artificialmente datos socioeconómicos inexistentes para ubicaciones desconocidas o fuera de término.




In [6]:
# Variables que describen específicamente a cada distrito
columnas_distrito = [
    'distrito_id',
    'codigo_postal_principal',
    'poblacion',
    'superficie_km2',
    'num_hospitales',
    'base_samur',
    'renta_media_hogar',
    'edad_media',
    'pct_mayores_65',
    'pct_extranjeros',
    'tasa_paro',
    'indice_vulnerabilidad',
    'num_centros_salud',
    'num_colegios',
    'num_comisarias',
    'parques_m2_habitante',
    'densidad_hab_km2'
]

# Nos quedamos solo con las columnas que existen en nuestra matriz
columnas_distrito = [
    col for col in columnas_distrito
    if col in matriz_x_mensual.columns
]

# Creamos una fila para OTROS en cada mes
filas_otros = []

for mes in matriz_x_mensual['mes'].unique():

    # Copiamos una fila del mes:
    # sus variables meteorológicas y de calendario son comunes
    fila = matriz_x_mensual[
        matriz_x_mensual['mes'] == mes
    ].iloc[[0]].copy()

    # Identificamos la nueva categoría
    fila['distrito'] = 'OTROS / DISTRITO NO IDENTIFICADO'

    # No inventamos características del distrito
    fila[columnas_distrito] = np.nan

    filas_otros.append(fila)

# Unimos las 12 filas nuevas a la matriz X
matriz_x_ampliada = pd.concat(
    [matriz_x_mensual] + filas_otros,
    ignore_index=True
)

# Comprobaciones
print("MATRIZ X AMPLIADA")
print("=" * 50)

print("Filas:", len(matriz_x_ampliada))
print("Meses:", matriz_x_ampliada['mes'].nunique())
print("Distritos:", matriz_x_ampliada['distrito'].nunique())
print("Columnas:", len(matriz_x_ampliada.columns))

print("\nFilas de OTROS:")
print(
    matriz_x_ampliada[
        matriz_x_ampliada['distrito'] == 'OTROS / DISTRITO NO IDENTIFICADO'
    ][['mes', 'distrito'] + columnas_distrito].to_string(index=False)
)

# Comprobamos que hay 12 meses x 22 categorías
assert len(matriz_x_ampliada) == 12 * 22
assert matriz_x_ampliada.groupby(
    ['mes', 'distrito']
).size().max() == 1

MATRIZ X AMPLIADA
Filas: 264
Meses: 12
Distritos: 22
Columnas: 57

Filas de OTROS:
       mes                         distrito  distrito_id  codigo_postal_principal  poblacion  superficie_km2  num_hospitales  base_samur  renta_media_hogar  edad_media  pct_mayores_65  pct_extranjeros  tasa_paro  indice_vulnerabilidad  num_centros_salud  num_colegios  num_comisarias  parques_m2_habitante  densidad_hab_km2
     ENERO OTROS / DISTRITO NO IDENTIFICADO          NaN                      NaN        NaN             NaN             NaN         NaN                NaN         NaN             NaN              NaN        NaN                    NaN                NaN           NaN             NaN                   NaN               NaN
   FEBRERO OTROS / DISTRITO NO IDENTIFICADO          NaN                      NaN        NaN             NaN             NaN         NaN                NaN         NaN             NaN              NaN        NaN                    NaN                NaN           NaN  

**Análisis**
* **Nueva dimensionalidad:** `matriz_x_ampliada` pasa de **252 a 264 filas** (12 meses × 22 categorías espaciales) manteniendo las **57 columnas**.
* **Coherencia de valores faltantes:** La inspección de las 12 filas de `'OTROS / DISTRITO NO IDENTIFICADO'` confirma que los 17 campos específicos de distrito contienen `NaN` de forma controlada, mientras que las 40 columnas restantes (identificadores temporales, meteorología y calendario) conservan sus valores mensuales completos.

### 4. Construcción y Validación del Dataset Completo ($X + y$)

Generamos la estructura cartesiana completa del proyecto cruzando las **264 combinaciones** `(mes, distrito)` de `matriz_x_ampliada` con las **5 categorías horarias** (`Madrugada`, `Mañana`, `Tarde`, `Noche` y `Desconocido`), obteniendo **1.320 filas**. Sobre esta base se incorporan las 57 variables explicativas ($X$) y los conteos de intervenciones ($y$), imputando con `0` las combinaciones sin actividad registrada y ejecutando una batería de validaciones estructurales (`assert`).

En los registros transaccionales de emergencias (`matriz_y_franjas_agrupada`), las combinaciones donde no hubo ningún incidente simplemente no existen como fila (solo aparecían 1.303 grupos). Si no se construyera explícitamente el producto cartesiano completo antes de unir $X$ e $y$, el modelo predictivo ignoraría los escenarios de demanda nula (`total_intervenciones = 0`), introduciendo un sesgo al alza en las predicciones.

In [7]:
# 1. Definimos las cinco categorías horarias
franjas = [
    '1. Madrugada (00:00 - 05:59)',
    '2. Mañana (06:00 - 11:59)',
    '3. Tarde (12:00 - 17:59)',
    '4. Noche (18:00 - 23:59)',
    'Desconocido'
]

# 2. Creamos todas las combinaciones posibles:
# mes x distrito x franja horaria
combinaciones = matriz_x_ampliada[
    ['mes', 'distrito']
].merge(
    pd.DataFrame({'franja_horaria': franjas}),
    how='cross'
)

# 3. Incorporamos las variables explicativas de X
dataset_final = combinaciones.merge(
    matriz_x_ampliada,
    on=['mes', 'distrito'],
    how='left',
    validate='many_to_one'
)

# 4. Incorporamos la variable objetivo y
dataset_final = dataset_final.merge(
    matriz_y_franjas_agrupada,
    on=['mes', 'distrito', 'franja_horaria'],
    how='left',
    validate='one_to_one'
)

# 5. Las combinaciones sin intervenciones reciben valor 0
dataset_final['total_intervenciones'] = (
    dataset_final['total_intervenciones'].fillna(0).astype(int)
)

# 6. Comprobaciones
print("DATASET FINAL")
print("=" * 55)

print("Número de filas:", len(dataset_final))
print("Número de columnas:", len(dataset_final.columns))
print("Meses:", dataset_final['mes'].nunique())
print("Distritos:", dataset_final['distrito'].nunique())
print("Franjas:", dataset_final['franja_horaria'].nunique())

print(
    "\nTotal de intervenciones:",
    dataset_final['total_intervenciones'].sum()
)

print(
    "Combinaciones con cero intervenciones:",
    (dataset_final['total_intervenciones'] == 0).sum()
)

print(
    "Valores nulos en la variable objetivo:",
    dataset_final['total_intervenciones'].isna().sum()
)

# 7. Verificamos la estructura esperada
assert len(dataset_final) == 1320
assert dataset_final['mes'].nunique() == 12
assert dataset_final['distrito'].nunique() == 22
assert dataset_final['franja_horaria'].nunique() == 5
assert dataset_final['total_intervenciones'].sum() == 146567
assert dataset_final['total_intervenciones'].isna().sum() == 0
assert not dataset_final.duplicated(
    ['mes', 'distrito', 'franja_horaria']
).any()

print("\n¡Todas las comprobaciones se han superado!")

# 8. Mostramos una pequeña muestra
display(dataset_final.head(10))

DATASET FINAL
Número de filas: 1320
Número de columnas: 59
Meses: 12
Distritos: 22
Franjas: 5

Total de intervenciones: 146567
Combinaciones con cero intervenciones: 17
Valores nulos en la variable objetivo: 0

¡Todas las comprobaciones se han superado!


,mes,distrito,franja_horaria,mes_num,distrito_id,temperatura_max_media,temperatura_max_absoluta,temperatura_min_media,temperatura_min_absoluta,temperatura_media,...,pct_mayores_65,pct_extranjeros,tasa_paro,indice_vulnerabilidad,num_centros_salud,num_colegios,num_comisarias,parques_m2_habitante,densidad_hab_km2,total_intervenciones
0,ENERO,CENTRO,1. Madrugada (00:00 - 05:59),1,1.0,9.674194,14.7,0.612903,-6.2,5.274194,...,22.1,21.5,8.2,0.35,8.0,25.0,3.0,3.2,28626.8,338
1,ENERO,CENTRO,2. Mañana (06:00 - 11:59),1,1.0,9.674194,14.7,0.612903,-6.2,5.274194,...,22.1,21.5,8.2,0.35,8.0,25.0,3.0,3.2,28626.8,313
2,ENERO,CENTRO,3. Tarde (12:00 - 17:59),1,1.0,9.674194,14.7,0.612903,-6.2,5.274194,...,22.1,21.5,8.2,0.35,8.0,25.0,3.0,3.2,28626.8,584
3,ENERO,CENTRO,4. Noche (18:00 - 23:59),1,1.0,9.674194,14.7,0.612903,-6.2,5.274194,...,22.1,21.5,8.2,0.35,8.0,25.0,3.0,3.2,28626.8,463
4,ENERO,CENTRO,Desconocido,1,1.0,9.674194,14.7,0.612903,-6.2,5.274194,...,22.1,21.5,8.2,0.35,8.0,25.0,3.0,3.2,28626.8,17
5,ENERO,ARGANZUELA,1. Madrugada (00:00 - 05:59),1,2.0,9.674194,14.7,0.612903,-6.2,5.274194,...,18.5,18.2,9.5,0.42,6.0,22.0,2.0,8.5,23974.5,44
6,ENERO,ARGANZUELA,2. Mañana (06:00 - 11:59),1,2.0,9.674194,14.7,0.612903,-6.2,5.274194,...,18.5,18.2,9.5,0.42,6.0,22.0,2.0,8.5,23974.5,160
7,ENERO,ARGANZUELA,3. Tarde (12:00 - 17:59),1,2.0,9.674194,14.7,0.612903,-6.2,5.274194,...,18.5,18.2,9.5,0.42,6.0,22.0,2.0,8.5,23974.5,250
8,ENERO,ARGANZUELA,4. Noche (18:00 - 23:59),1,2.0,9.674194,14.7,0.612903,-6.2,5.274194,...,18.5,18.2,9.5,0.42,6.0,22.0,2.0,8.5,23974.5,183
9,ENERO,ARGANZUELA,Desconocido,1,2.0,9.674194,14.7,0.612903,-6.2,5.274194,...,18.5,18.2,9.5,0.42,6.0,22.0,2.0,8.5,23974.5,8



**Análisis**
* **Dimensiones finales exactas:** El dataset consolidado cuenta con **1.320 filas** ($12\text{ meses} \times 22\text{ distritos} \times 5\text{ franjas}$) y **59 columnas** (las 3 claves primarias, 55 variables explicativas y la variable objetivo `total_intervenciones`).
* **Imputación de ceros reales:** Al comparar las **1.320 combinaciones posibles** frente a los **1.303 grupos con actividad** del paso anterior, se identifican exactamente **17 combinaciones con cero intervenciones** (correspondientes casi en su totalidad a la franja `Desconocido` o a la categoría residual de distrito en meses puntuales), que quedan correctamente registradas con valor `0` y sin nulos en `y`.
* **Integridad y coherencia en la muestra:** La suma total de `total_intervenciones` se mantiene en **146.567 casos** sin duplicados. En la muestra de enero se aprecia claramente el patrón intradiario dentro de un mismo distrito: por ejemplo, en `CENTRO`, las intervenciones pasan de **338** en madrugada y **313** en mañana a un pico de **584** por la tarde y **463** por la noche (más **17** sin hora registrada), replicándose una curva similar en `ARGANZUELA` (pico de **250** por la tarde frente a **44** de madrugada).

### 5. Auditoría de Calidad e Integridad del Dataset Final

Inspección sobre `dataset_final` antes de exportarlo para verificar cuatro aspectos críticos:
1. **Mapa de valores ausentes:** Se contabilizan los `NaN` por columna y se comprueba si coinciden exactamente con las filas de `'OTROS / DISTRITO NO IDENTIFICADO'`.
2. **Peso de la categoría residual:** Se cuantifican las filas e intervenciones agrupadas bajo distrito no identificado.
3. **Consistencia de la distribución horaria:** Se verifica que los totales por franja coincidan exactamente con los calculados en el Paso 3 antes de los cruces.
4. **Diagnóstico de ceros imputados:** Se listan explícitamente las 17 combinaciones `(mes, distrito, franja_horaria)` que tuvieron `0` intervenciones para confirmar que responden a la realidad operativa y no a errores de unión.

En un pipeline de ingeniería de datos, un cruce mal planteado puede introducir valores nulos ocultos en registros válidos o alterar silenciosamente los conteos agregados. Esta auditoría final actúa como cierre de control de calidad, certificando que cada valor faltante o cero tiene una explicación trazable y documentada antes de pasar a la fase de análisis exploratorio (EDA) y modelado.


In [8]:
# 1. Número de valores ausentes por columna
nulos = dataset_final.isna().sum()

print("VALORES AUSENTES POR COLUMNA")
print("=" * 55)

print(nulos[nulos > 0].sort_values(ascending=False))

# 2. Comprobamos los valores ausentes de OTROS
otros = dataset_final[
    dataset_final['distrito'] == 'OTROS / DISTRITO NO IDENTIFICADO'
]

print("\nCATEGORÍA OTROS")
print("=" * 55)

print("Filas:", len(otros))
print("Intervenciones:", otros['total_intervenciones'].sum())

# 3. Distribución de intervenciones por franja horaria
print("\nINTERVENCIONES POR FRANJA")
print("=" * 55)

print(
    dataset_final.groupby('franja_horaria')['total_intervenciones']
    .sum()
)

# 4. Combinaciones con cero intervenciones
ceros = dataset_final[
    dataset_final['total_intervenciones'] == 0
]

print("\nCOMBINACIONES CON CERO INTERVENCIONES")
print("=" * 55)

print(
    ceros[['mes', 'distrito', 'franja_horaria']]
    .to_string(index=False)
)

# 5. Comprobaciones finales
print("\nCOMPROBACIONES")
print("=" * 55)

print("Total de intervenciones:", dataset_final['total_intervenciones'].sum())
print("Total de filas:", len(dataset_final))
print("Filas duplicadas:", dataset_final.duplicated(
    ['mes', 'distrito', 'franja_horaria']
).sum())

assert dataset_final['total_intervenciones'].sum() == 146567
assert len(dataset_final) == 1320

VALORES AUSENTES POR COLUMNA
distrito_id                60
codigo_postal_principal    60
poblacion                  60
superficie_km2             60
num_hospitales             60
base_samur                 60
renta_media_hogar          60
edad_media                 60
pct_mayores_65             60
pct_extranjeros            60
tasa_paro                  60
indice_vulnerabilidad      60
num_centros_salud          60
num_colegios               60
num_comisarias             60
parques_m2_habitante       60
densidad_hab_km2           60
dtype: int64

CATEGORÍA OTROS
Filas: 60
Intervenciones: 601

INTERVENCIONES POR FRANJA
franja_horaria
1. Madrugada (00:00 - 05:59)    20562
2. Mañana (06:00 - 11:59)       31007
3. Tarde (12:00 - 17:59)        50621
4. Noche (18:00 - 23:59)        42955
Desconocido                      1422
Name: total_intervenciones, dtype: int64

COMBINACIONES CON CERO INTERVENCIONES
       mes                         distrito            franja_horaria
     MARZO         


**Análisis**
* **Ausencia total de nulos inesperados:** Únicamente las 17 variables territoriales presentan valores faltantes, y en todas ellas el conteo es exactamente **60** ($12\text{ meses} \times 5\text{ franjas horarias}$), que corresponde a las 60 filas de `'OTROS / DISTRITO NO IDENTIFICADO'` (**601 intervenciones**, el 0,41 % del total). Los 21 distritos oficiales tienen el **100 % de sus variables informadas**.
* **Invariabilidad de la distribución horaria:** Los totales por franja (`20.562` madrugada, `31.007` mañana, `50.621` tarde, `42.955` noche y `1.422` desconocido) suman exactamente **146.567 intervenciones**, demostrando que las operaciones de ampliación y *merge* no han alterado ni un solo registro.
* **Naturaleza lógica de las 17 combinaciones con cero intervenciones:**
  * **16 de los 17 ceros** pertenecen a la franja `'Desconocido'` (meses en los que todos los avisos de distritos como `MORATALAZ`, `VILLA DE VALLECAS`, `BARAJAS`, `VICALVARO` o `OTROS` tuvieron su hora de llamada correctamente registrada).
  * **El único cero en una franja horaria real** ocurre en `AGOSTO` para `'OTROS / DISTRITO NO IDENTIFICADO'` en la franja `'2. Mañana (06:00 - 11:59)'`, algo totalmente coherente dado el bajo volumen mensual de esa categoría residual (media de ~10 casos por franja y mes) y la menor actividad veraniega en agosto.
  * **Ningún distrito oficial tiene cero intervenciones** en ninguna de las cuatro franjas horarias reales durante todo 2024.

#### 5.1. Segregación y Auditoría Matemática de Valores Ausentes

Dividimos `dataset_final` en dos subconjuntos (`dataset_oficial` con los 21 distritos administrativos y `dataset_otros` con la categoría residual `'OTROS / DISTRITO NO IDENTIFICADO'`) y se ejecuta una comprobación exacta sobre el total de celdas nulas en cada bloque mediante aserciones (`assert`).

Para aislar los datos territoriales completos de los registros sin ubicación asignada y certificar matemáticamente que el 100 % de los valores `NaN` del dataset están acotados a las 17 variables específicas de distrito dentro de la categoría residual, garantizando que `dataset_oficial` está completamente limpio y listo para entrenar modelos predictivos que no admitan valores ausentes.


In [9]:
# Separamos los distritos oficiales y la categoría OTROS

dataset_oficial = dataset_final[
    dataset_final['distrito'] != 'OTROS / DISTRITO NO IDENTIFICADO'
]

dataset_otros = dataset_final[
    dataset_final['distrito'] == 'OTROS / DISTRITO NO IDENTIFICADO'
]

print("AUDITORÍA FINAL DE VALORES AUSENTES")
print("=" * 55)

print("\nDistritos oficiales:")
print("Filas:", len(dataset_oficial))
print("Valores nulos:", dataset_oficial.isna().sum().sum())

print("\nCategoría OTROS:")
print("Filas:", len(dataset_otros))
print("Valores nulos:", dataset_otros.isna().sum().sum())

# Comprobamos que los distritos oficiales no tienen nulos
assert dataset_oficial.isna().sum().sum() == 0

# Comprobamos que los nulos de OTROS son únicamente
# las características específicas del distrito
assert dataset_otros.isna().sum().sum() == 17 * 60

print("\n¡Auditoría superada!")

AUDITORÍA FINAL DE VALORES AUSENTES

Distritos oficiales:
Filas: 1260
Valores nulos: 0

Categoría OTROS:
Filas: 60
Valores nulos: 1020

¡Auditoría superada!


**Análisis**
* **Subconjunto de distritos oficiales (`dataset_oficial`):** Cuenta con exactamente **1.260 filas** ($12\text{ meses} \times 21\text{ distritos} \times 5\text{ franjas}$) y **0 valores nulos** en todo el bloque, confirmando una matriz 100 % densa e íntegra para los 21 distritos de Madrid.
* **Subconjunto residual (`dataset_otros`):** Contiene **60 filas** ($12\text{ meses} \times 5\text{ franjas}$) y exactamente **1.020 valores nulos**, cifra que coincide al milímetro con el producto de las **17 variables territoriales** multiplicadas por las **60 filas** ($17 \times 60 = 1.020$), demostrando que ni una sola variable meteorológica, de calendario o de intervenciones tiene valores faltantes.

### 6. Ingeniería de Variables (*Feature Engineering*)

#### 6.1. Indicadores Climáticos Extremos y Amplitud Térmica

Partiendo de los registros meteorológicos diarios originales, se construyen 6 nuevas variables agregadas a nivel mensual:
* `rango_termico_medio`: promedio mensual de la amplitud térmica diaria ($\text{T}_{\max} - \text{T}_{\min}$).
* `dias_calor_extremo`: conteo de días con temperatura máxima $\ge 35\text{ °C}$.
* `dias_frio`: conteo de días con heladas o temperatura mínima $\le 0\text{ °C}$.
* `dias_lluvia`: conteo de días con precipitación efectiva ($> 0\text{ mm}$).
* `dias_viento_fuerte`: conteo de días con rachas máximas superiores a $40\text{ km/h}$.
* `precipitacion_media_diaria`: promedio diario de precipitación (en mm) durante el mes.

**¿Por qué se hace?**
Las medias mensuales suavizan los picos diarios y pueden ocultar eventos climáticos severos que disparan las emergencias médicas o accidentes (como olas de calor, heladas o jornadas de lluvia intensa). Capturar umbrales extremos y la oscilación térmica diaria aporta señales no lineales de alto valor predictivo.

In [10]:
# Trabajamos con una copia de la meteorología original
meteo = datasets['Meteorología'].copy()

# Convertimos la fecha
meteo['fecha'] = pd.to_datetime(
    meteo['fecha'],
    format='%d-%m-%Y',
    errors='coerce'
)

# Comprobamos que las fechas se han convertido correctamente
assert meteo['fecha'].notna().all()

# Creamos el número de mes
meteo['mes_num'] = meteo['fecha'].dt.month

# 1. Rango térmico diario
meteo['rango_termico_diario'] = (
    meteo['temperatura_max'] - meteo['temperatura_min']
)

# 2. Indicadores diarios de fenómenos meteorológicos
meteo['dia_calor_extremo'] = (
    meteo['temperatura_max'] >= 35
).astype(int)

meteo['dia_frio'] = (
    meteo['temperatura_min'] <= 0
).astype(int)

meteo['dia_lluvioso'] = (
    meteo['precipitacion_mm'] > 0
).astype(int)

meteo['dia_viento_fuerte'] = (
    meteo['viento_max_kmh'] > 40
).astype(int)

# 3. Agregamos las nuevas variables por mes
meteo_mensual_nuevo = (
    meteo.groupby('mes_num')
    .agg(
        rango_termico_medio=('rango_termico_diario', 'mean'),
        dias_calor_extremo=('dia_calor_extremo', 'sum'),
        dias_frio=('dia_frio', 'sum'),
        dias_lluvia=('dia_lluvioso', 'sum'),
        dias_viento_fuerte=('dia_viento_fuerte', 'sum'),
        precipitacion_media_diaria=('precipitacion_mm', 'mean')
    )
    .reset_index()
)

print("NUEVAS VARIABLES METEOROLÓGICAS")
print("=" * 55)

display(meteo_mensual_nuevo)

# Comprobaciones
assert len(meteo_mensual_nuevo) == 12
assert meteo_mensual_nuevo.isna().sum().sum() == 0

NUEVAS VARIABLES METEOROLÓGICAS


,mes_num,rango_termico_medio,dias_calor_extremo,dias_frio,dias_lluvia,dias_viento_fuerte,precipitacion_media_diaria
0,1,9.061290,0,13,11,1,1.919355
1,2,8.725147,0,6,9,1,1.022414
2,3,10.706615,0,0,7,0,0.793548
3,4,10.306667,1,0,7,0,0.800000
4,5,9.546491,13,0,8,0,1.154839
5,6,11.346667,28,0,13,1,1.456667
6,7,9.844925,31,0,7,1,0.251613
7,8,8.943053,23,0,7,0,0.117742
8,9,9.320000,5,0,7,0,0.370000
9,10,10.222581,0,0,9,0,0.956452



**Análisis**
* Se genera una tabla limpia de **12 filas** (una por mes) con **0 valores nulos**.
* **Estacionalidad térmica extrema:** Los días de calor extremo ($\ge 35\text{ °C}$) se concentran exclusivamente en verano (**13 días** en junio, **28** en julio, **31** en agosto y **23** en septiembre), mientras que los días de frío ($\le 0\text{ °C}$) aparecen en invierno y primavera temprana (**13 días** en enero, **6** en febrero, **11** en marzo y **1** en abril).
* **Amplitud y lluvia:** El rango térmico medio se mantiene elevado en todo el año (entre **8,73 °C** en febrero y **11,35 °C** en junio), típico del clima continental de Madrid. La precipitación media diaria alcanza su máximo en enero (**1,92 mm/día**) y noviembre (**1,69 mm/día**), cayendo a mínimos en agosto (**0,12 mm/día**).

#### 6.2. Variables Temporales Cíclicas y Proporciones de Calendario

Creamos `dataset_modelado` a partir de `dataset_final`, se integran las 6 nuevas variables meteorológicas y se generan 6 nuevas características temporales:
* **Codificación trigonométrica del mes (`mes_sin`, `mes_cos`):** Transformación circular de `mes_num` mediante seno y coseno ($\sin(2\pi \cdot \frac{\text{mes}-1}{12})$ y $\cos(2\pi \cdot \frac{\text{mes}-1}{12})$).
* **Tasas relativas de calendario (`pct_festivos`, `pct_fines_semana`, `pct_dias_evento`, `pct_vacaciones`):** Porcentaje de días del mes que corresponden a cada categoría respecto al total de días (`dias_mes`).

**¿Por qué se hace?**
* **Continuidad temporal:** Si un modelo interpreta `mes_num` como un número lineal, asume que diciembre (`12`) y enero (`1`) están en extremos opuestos, cuando en realidad son consecutivos. Las componentes `mes_sin` y `mes_cos` preservan la proximidad circular entre ambos meses.
* **Normalización por duración del mes:** Dado que los meses tienen distinta duración (29 días en febrero de 2024 frente a 31 en enero o marzo), expresar los festivos, fines de semana o eventos en porcentaje elimina el sesgo de escala por número de días.

In [11]:
# Creamos una copia del dataset 
dataset_modelado = dataset_final.copy()

# Incorporamos las nuevas variables meteorológicas
dataset_modelado = dataset_modelado.merge(
    meteo_mensual_nuevo,
    on='mes_num',
    how='left',
    validate='many_to_one'
)

# 1. Variables cíclicas del mes
dataset_modelado['mes_sin'] = np.sin(
    2 * np.pi * (dataset_modelado['mes_num'] - 1) / 12
)

dataset_modelado['mes_cos'] = np.cos(
    2 * np.pi * (dataset_modelado['mes_num'] - 1) / 12
)

# 2. Proporciones de calendario
dataset_modelado['pct_festivos'] = (
    dataset_modelado['dias_festivos'] /
    dataset_modelado['dias_mes'] * 100
)

dataset_modelado['pct_fines_semana'] = (
    dataset_modelado['dias_fin_semana'] /
    dataset_modelado['dias_mes'] * 100
)

dataset_modelado['pct_dias_evento'] = (
    dataset_modelado['dias_con_evento'] /
    dataset_modelado['dias_mes'] * 100
)

dataset_modelado['pct_vacaciones'] = (
    dataset_modelado['dias_vacaciones'] /
    dataset_modelado['dias_mes'] * 100
)

print("Variables temporales incorporadas correctamente.")

Variables temporales incorporadas correctamente.


#### 6.3. Ratios de Cobertura Sanitaria e Interacciones Espacio-Temporales

Se construyen 5 variables derivadas que combinan atributos demográficos, recursos sanitarios y factores climáticos:
* `poblacion_mayor_65`: volumen absoluto estimado de habitantes mayores de 65 años en cada distrito.
* `hospitales_por_100k`: dotación hospitalaria relativa por cada 100.000 habitantes.
* `poblacion_por_base_samur`: carga poblacional asignada a cada base SAMUR (asignando `NaN` cuando el distrito no dispone de base propia para evitar divisiones por cero).
* `riesgo_calor_mayores`: término de interacción entre el porcentaje de población mayor de 65 años y los días de calor extremo del mes.
* `riesgo_lluvia_densidad`: término de interacción entre la densidad poblacional ($\text{hab/km}^2$) y los días de lluvia del mes.

**¿Por qué se hace?**
Los recuentos brutos de hospitales o bases SAMUR no reflejan la presión asistencial real si no se relativizan respecto al tamaño de la población del distrito. Por su parte, los términos de interacción multiplicativos ayudan especialmente a los modelos lineales a capturar efectos compuestos (por ejemplo, cómo una ola de calor afecta con mayor intensidad a distritos envejecidos, o cómo la lluvia incrementa la siniestralidad vial en zonas densamente pobladas).

In [12]:
# 1. Población estimada mayor de 65 años
dataset_modelado['poblacion_mayor_65'] = (
    dataset_modelado['poblacion'] *
    dataset_modelado['pct_mayores_65'] / 100
)

# 2. Hospitales por cada 100.000 habitantes
dataset_modelado['hospitales_por_100k'] = (
    dataset_modelado['num_hospitales'] /
    dataset_modelado['poblacion'] * 100000
)

# 3. Población por base SAMUR
dataset_modelado['poblacion_por_base_samur'] = (
    dataset_modelado['poblacion'] /
    dataset_modelado['base_samur'].replace(0, np.nan)
)

# 4. Interacción entre población mayor y calor extremo
dataset_modelado['riesgo_calor_mayores'] = (
    dataset_modelado['pct_mayores_65'] *
    dataset_modelado['dias_calor_extremo']
)

# 5. Interacción entre densidad y lluvia
dataset_modelado['riesgo_lluvia_densidad'] = (
    dataset_modelado['densidad_hab_km2'] *
    dataset_modelado['dias_lluvia']
)

print("Variables sociodemográficas e interacciones creadas.")

Variables sociodemográficas e interacciones creadas.


#### 6.4. Auditoría Estadística y de Nulos de las Nuevas Variables

Verificamos la dimensionalidad de `dataset_modelado`, se calculan los estadísticos descriptivos (media, desviación típica, mínimos, cuartiles y máximos) de las 17 nuevas variables creadas y se audita la distribución de valores ausentes (`NaN`).

In [13]:
# Variables nuevas creadas
variables_nuevas = [
    'rango_termico_medio',
    'dias_calor_extremo',
    'dias_frio',
    'dias_lluvia',
    'dias_viento_fuerte',
    'precipitacion_media_diaria',
    'mes_sin',
    'mes_cos',
    'pct_festivos',
    'pct_fines_semana',
    'pct_dias_evento',
    'pct_vacaciones',
    'poblacion_mayor_65',
    'hospitales_por_100k',
    'poblacion_por_base_samur',
    'riesgo_calor_mayores',
    'riesgo_lluvia_densidad'
]

print("AUDITORÍA DE INGENIERÍA DE VARIABLES")
print("=" * 55)

print("Filas:", len(dataset_modelado))
print("Columnas:", len(dataset_modelado.columns))
print("Nuevas variables:", len(variables_nuevas))

print("\nNuevas variables:")
print(dataset_modelado[variables_nuevas].describe().round(2).T)

print("\nValores nulos en las nuevas variables:")
print(dataset_modelado[variables_nuevas].isna().sum())

# Verificaciones fundamentales
assert len(dataset_modelado) == 1320
assert dataset_modelado['total_intervenciones'].sum() == 146567
assert dataset_modelado['distrito'].nunique() == 22

print("\n¡Ingeniería de variables completada!")

AUDITORÍA DE INGENIERÍA DE VARIABLES
Filas: 1320
Columnas: 76
Nuevas variables: 17

Nuevas variables:
                             count       mean       std       min        25%  \
rango_termico_medio         1320.0       9.86      0.77      8.73       9.26   
dias_calor_extremo          1320.0       8.42     11.62      0.00       0.00   
dias_frio                   1320.0       2.50      4.58      0.00       0.00   
dias_lluvia                 1320.0       8.17      1.95      6.00       7.00   
dias_viento_fuerte          1320.0       0.33      0.47      0.00       0.00   
precipitacion_media_diaria  1320.0       0.96      0.53      0.12       0.69   
mes_sin                     1320.0       0.00      0.71     -1.00      -0.59   
mes_cos                     1320.0      -0.00      0.71     -1.00      -0.59   
pct_festivos                1320.0       3.51      3.36      0.00       0.00   
pct_fines_semana            1320.0      28.43      2.51     25.81      25.81   
pct_dias_evento   


**Análisis**
* **Crecimiento:** El dataset alcanza **1.320 filas y 76 columnas** (59 originales + 17 nuevas), manteniendo intactas las **146.567 intervenciones** y las **22 categorías espaciales**.
* **Trazabilidad de valores nulos:**
  * Las **12 variables meteorológicas y temporales** (`rango_termico_medio` hasta `pct_vacaciones`) presentan **0 valores nulos** en las 1.320 filas.
  * Las **4 variables territoriales e interacciones** (`poblacion_mayor_65`, `hospitales_por_100k`, `riesgo_calor_mayores`, `riesgo_lluvia_densidad`) tienen exactamente **60 nulos**, correspondientes exclusivamente a `'OTROS / DISTRITO NO IDENTIFICADO'`.
  * **`poblacion_por_base_samur` presenta 660 nulos (50 % de las filas):** Este resultado es totalmente coherente y esperado. Al los 60 nulos de `OTROS` se suman **600 nulos** correspondientes a **10 distritos oficiales que no cuentan con base SAMUR propia** ($10\text{ distritos} \times 12\text{ meses} \times 5\text{ franjas} = 600\text{ filas}$), donde `base_samur == 0` fue reemplazado intencionadamente por `NaN` para evitar divisiones infinitas (`inf`). 

#### 6.5.Eliminación Variable con Alta Nulidad

Se elimina la variable `poblacion_por_base_samur` de `dataset_modelado` mediante `.drop(columns=...)` y se comprueba tanto la nueva dimensionalidad de la tabla como el recuento de valores ausentes en las variables sociodemográficas e interacciones restantes.

**¿Por qué se hace?**
En la auditoría anterior se detectó que `poblacion_por_base_samur` presentaba **660 valores nulos** (el 50 % del dataset), ya que casi la mitad de los distritos oficiales de Madrid no cuentan con una base SAMUR ubicada físicamente dentro de su perímetro. Dado que el servicio SAMUR opera en red cubriendo distritos colindantes y la variable original `base_samur` ya recoge de forma limpia la presencia o ausencia de base (`1` o `0`), eliminar este ratio evita introducir un 50 % de datos ausentes que obligarían a descartar la mitad de los distritos en modelos sensibles a `NaN` o a realizar imputaciones artificiales.

El dataset pasa de **76 a 75 columnas**, conservando intactas las **1.320 filas**, y la comprobación confirma que `poblacion_por_base_samur` ha sido eliminada (`False`).


In [14]:
# Eliminar la variable creada
dataset_modelado = dataset_modelado.drop(
    columns=['poblacion_por_base_samur']
)

# Comprobar resultado
print("Dimensiones:", dataset_modelado.shape)
print("¿Sigue existiendo la columna?",
      'poblacion_por_base_samur' in dataset_modelado.columns)

print("\nValores nulos restantes en las variables socioeconómicas:")
display(
    dataset_modelado[
        ['poblacion_mayor_65', 'hospitales_por_100k',
         'riesgo_calor_mayores', 'riesgo_lluvia_densidad']
    ].isna().sum()
)

Dimensiones: (1320, 75)
¿Sigue existiendo la columna? False

Valores nulos restantes en las variables socioeconómicas:


poblacion_mayor_65        60
hospitales_por_100k       60
riesgo_calor_mayores      60
riesgo_lluvia_densidad    60
dtype: int64

### 7. Validación Final y Exportación del Dataset Definitivo
Última verificación global sobre `dataset_modelado` (dimensiones, suma de intervenciones, cardinalidad de claves, ausencia de duplicados y nulos en la variable objetivo) y se exporta la tabla consolidada en formato CSV bajo el nombre `dataset_samur_modelado.csv`.


In [15]:
# Comprobar dataset final de modelado
print("DIMENSIONES:", dataset_modelado.shape)
print("TOTAL INTERVENCIONES:", dataset_modelado['total_intervenciones'].sum())
print("DISTRITOS:", dataset_modelado['distrito'].nunique())
print("MESES:", dataset_modelado['mes_num'].nunique())
print("FRANJAS:", dataset_modelado['franja_horaria'].nunique())
print("DUPLICADOS:", dataset_modelado.duplicated(
    subset=['mes_num', 'distrito', 'franja_horaria']
).sum())
print("NULOS EN TARGET:", dataset_modelado['total_intervenciones'].isna().sum())

# Guardar versión definitiva
dataset_modelado.to_csv(
    'dataset_samur_modelado.csv',
    index=False,
    encoding='utf-8-sig'
)

DIMENSIONES: (1320, 75)
TOTAL INTERVENCIONES: 146567
DISTRITOS: 22
MESES: 12
FRANJAS: 5
DUPLICADOS: 0
NULOS EN TARGET: 0


In [16]:
import os

# Crear la carpeta si todavía no existe
os.makedirs('datasets limpios', exist_ok=True)

# Guardar el dataset definitivo
dataset_modelado.to_csv(
    os.path.join('datasets limpios', 'dataset_samur_modelado.csv'),
    index=False,
    encoding='utf-8-sig'
)

print("Dataset guardado correctamente.")
print("Ruta:", os.path.abspath(
    'datasets limpios/dataset_samur_modelado.csv'
))

Dataset guardado correctamente.
Ruta: c:\Users\bario\OneDrive\Documentos\UC3M\PROYECTO EN CIENCIA DE DATOS\proyecto_emergencias-20260915T155207Z-1-001\proyecto_emergencias\datasets limpios\dataset_samur_modelado.csv
